# 13 · Full-query reliability evaluation and group-aware validation

**Main report:** how accurately do the saved ML models predict histogram error on ALL test queries, and how do SA and
ML rank unreliable answers? Acceptance gates and fixed coverage are downstream diagnostics, not the whole evaluation.

## Existing results: report only (default)

1. Keep `execution_mode=report`, leave `report_run_dir` blank and Run All. Training and training-only installation/plots
   are skipped. The last cell selects the latest completed **13** result under your personal `output_root` and prints it.
2. Alternatively, run **only the final full-query report cell**, even after a Python restart. It creates/reads its own
   report widgets and does not import LightGBM or require any earlier training cell. Set `report_run_dir` to a complete
   saved 13 result directory when you want an explicit source. `run_dir` is used only by training and points to 12.
3. Normally leave `rows_file` blank. SA scores come from the hash-verified original logger recorded by 13. If it moved,
   supply that matching file. Missing SA inputs produce an explicit partial report; ML still uses the FULL test cohort.
4. Send **`full_query_report.zip`** from the new `full-query-report-...` directory. Saved models/results are not modified.

The report includes full-test numerical ML error, underestimation, AUC/Spearman, complete risk–coverage curves and the
original calibration-selected gates. Raw SA scores are not assumed to be relative-error percentages, so their numerical
MAE is undefined. Plot samples never replace full-query metrics. Query-time and E2E costs for these models remain unmeasured.

## Optional: train the validation-policy comparison again

Set `execution_mode=train` only to retrain the original controlled experiment below. Then use `run_dir` as described here.

**One controlled question:** does choosing the early-stopping iteration on held-out source-cell groups improve
reliability prediction on the existing outer cell test, compared with the current query validation policy?

Notebook 12 found a compact-information benefit on random query splits, but poor accepted-answer quality on held-out
cell groups. This notebook compares **validation policies**, not new feature sets, model families or gate algorithms.
The outer train/calibration/test split stays exactly as saved in notebook 12. Grouped validation necessarily also changes
the observations used for fitting; their counts and identities are reported. Neither policy refits on all outer train rows.

### Training inputs

1. Pull this branch in your personal Git folder. Run the configuration cell, choose `execution_mode=train`, then run the
   conditional installation cell. The default report mode does not install training dependencies.
2. Normally **leave `run_dir` blank**. The latest completed **notebook 12** run containing the requested seeds and cell
   splits under your personal `output_root` is selected by modification time and printed. To reproduce a particular run,
   enter that complete result directory. Do not enter an 08d directory or a ZIP here.
3. Keep `seeds=0,1,2`. Base histogram rates, tree budget, thread count and other LightGBM settings are inherited from 12.
   There is no merge-rate or model-search widget in this experiment. For a small first check you may request seed `0` only.
4. Run All. Both policies are retrained for `merged_only`, `compact16`, and `full_source` (18 fits for three seeds).
   Existing models and results are preserved; each attempt creates a new `group-validation-...` directory.

The completed 12 directory and its original 08d input directory must remain accessible. `data_dir` and `histogram_dir`
are optional relocation overrides for the original 08d files; their hashes must still match. No personal path is hard-coded.

In [ ]:
import ast
import base64
import hashlib
import importlib
import importlib.metadata
import json
import math
import os
import platform
import shutil
import tempfile
import time
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4
from zipfile import ZipFile, ZIP_DEFLATED

import numpy as np
import pandas as pd


def notebook_defaults():
    try:
        context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        root = context.notebookPath().get().split('/ml/notebooks/')[0]
        repo = root if root.startswith('/Workspace/') else '/Workspace' + root
        output = '/Workspace/Users/' + context.userName().get() + '/synopsislake-results'
    except (NameError, AttributeError):
        repo, output = str(Path.cwd()), str(Path(tempfile.gettempdir())/'synopsislake-results')
    return dict(execution_mode='report',repo_root=repo,output_root=output,run_dir='',
                report_run_dir='',rows_file='',data_dir='',histogram_dir='',seeds='0,1,2')


DEFAULTS = notebook_defaults()
try:
    for key, value in DEFAULTS.items():
        dbutils.widgets.text(key, value)
    get_config = dbutils.widgets.get
except NameError:
    def get_config(key):
        return os.environ.get(key.upper(), DEFAULTS[key])


VARIANTS = ('merged_only', 'compact16', 'full_source')
POLICIES = ('query', 'cell')
INNER_FRACTION = .10
EARLY_STOPPING_ROUNDS = 100


def load_settings():
    cfg = {key: str(get_config(key)).strip() for key in DEFAULTS}
    cfg['seeds'] = [int(x) for x in cfg['seeds'].split(',')]
    if not cfg['seeds'] or len(set(cfg['seeds'])) != len(cfg['seeds']):
        raise ValueError('Use distinct seeds from the completed notebook 12 run')
    if cfg['execution_mode'] not in {'report','train'}:
        raise ValueError('execution_mode must be report or train')
    return cfg

In [ ]:
# INSTALL_TRAINING_DEPENDENCIES
if str(get_config('execution_mode')).strip()=='train':
    get_ipython().run_line_magic('pip','install lightgbm==4.6.0')
else:
    print('Report mode: training dependency installation skipped.')

## Reuse existing implementation

Only pinned definition cells from the repository's notebook 12 are loaded into an isolated namespace. Its widgets,
training and plotting cells are never executed. Hashes use selected Python AST definitions so normal saved outputs,
formatting, notebook metadata and missing cell IDs do not cause a mismatch. Changed definitions fail explicitly.
When notebook files are not readable through the workspace filesystem, a read-only Jupyter export uses the signed-in
Databricks runtime identity. No personal Databricks token is embedded or requested.

Existing merged/source feature code, encoded float16 metadata, cohort rules, query parsing, error labels, estimate parity
and empirical calibration are reused unchanged. The preparation stage still constructs auxiliary 12 artifacts, but this
study trains only three variants and makes no runtime or storage claim.

In [ ]:
REUSE_SPECS = {'compact-04': None, 'compact-07': ['sha', 'save_json', 'choose_ids', 'resolve_source', 'import_existing_helpers'], 'compact-09': ['histogram_context', 'base_query', 'source_query', 'extract_query', 'frame_for_context'], 'compact-11': ['prepare_artifacts'], 'compact-13': ['evaluate_score', 'fixed_coverage']}
REUSE_AST_SHA256 = {'compact-04': '1ac8d26a90d6e402c1223960b898fee17f79e9cea33c7656a7b699c6b30da8d3', 'compact-07': '59ed8ee9fe9aa4d989d47bcdad2c42af5d8067ad777db1765b0a978884496d30', 'compact-09': '0b24e98832f4b742419f0a417f021533793f5dd850873d87ee4dae86b520df04', 'compact-11': '97c22976f16bf0223ec1f7053c1f0fa106a06f6f3d0f5620aee9dd401c0fb6de', 'compact-13': '98478f1eec1edc83eb37b3fa686fafa933c62ad85356842c86fec6c07440128f'}


def file_sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()


def save_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix+'.tmp')
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False, default=str))
    temporary.replace(path)


def selected_ast(document, cell_id, names):
    # Databricks exports can remove/rewrite cell IDs. Locate verified content instead of relying on IDs.
    matches = []
    for cell in document['cells']:
        if cell['cell_type'] != 'code':
            continue
        try:
            tree = ast.parse(''.join(cell['source']))
        except SyntaxError:
            continue  # Notebook magic is not a definition source.
        if names is not None:
            nodes = [node for node in tree.body if isinstance(node, ast.FunctionDef) and node.name in names]
            if {node.name for node in nodes} != set(names):
                continue
            tree = ast.Module(body=nodes, type_ignores=[])
        else:
            allowed = (ast.Import, ast.ImportFrom, ast.Assign, ast.FunctionDef)
            if any(not isinstance(node, allowed) and not
                   (isinstance(node, ast.Expr) and isinstance(node.value, ast.Constant) and isinstance(node.value.value, str))
                   for node in tree.body):
                continue
        digest = hashlib.sha256(ast.dump(tree, include_attributes=False).encode()).hexdigest()
        if digest == REUSE_AST_SHA256[cell_id]:
            matches.append(tree)
    if len(matches) != 1:
        raise ValueError('Missing, changed or duplicate notebook 12 definitions: '+cell_id+'. Pull matching notebooks 12 and 13.')
    return matches[0]


def export_workspace_notebook(path):
    # Some Databricks runtimes expose Git notebooks only through the Workspace API.
    # Use the signed-in runtime credentials; no personal token is stored in the notebook.
    from databricks.sdk import WorkspaceClient
    from databricks.sdk.service.workspace import ExportFormat
    client = WorkspaceClient()
    workspace_path = str(path)
    if workspace_path.startswith('/Workspace/'):
        workspace_path = workspace_path[len('/Workspace'):]
    candidates = [workspace_path]
    if workspace_path.endswith('.ipynb'):
        candidates.append(workspace_path[:-len('.ipynb')])
    errors = []
    for candidate in candidates:
        try:
            exported = client.workspace.export(path=candidate, format=ExportFormat.JUPYTER)
            if not exported.content:
                raise ValueError('Workspace export returned no content')
            return base64.b64decode(exported.content)
        except Exception as error:
            errors.append(f'{candidate}: {type(error).__name__}: {error}')
    raise ValueError('Cannot export notebook 12 with the runtime identity: '+'; '.join(errors))


def read_notebook_document(path):
    path = Path(path)
    try:
        raw = path.read_bytes()
        document = json.loads(raw)
        method = 'workspace filesystem'
    except (OSError, ValueError, UnicodeError):
        if not str(path).startswith('/Workspace/'):
            raise
        try:
            raw = export_workspace_notebook(path)
            document = json.loads(raw)
            method = 'Workspace API Jupyter export using runtime identity'
        except Exception as error:
            raise ValueError('Cannot read notebook 12 as a file or Workspace export. Check repo_root and notebook access. '+str(error)) from error
    return document, dict(path=str(path),read_method=method,document_sha256=hashlib.sha256(raw).hexdigest())


def load_reused_definitions(repo):
    path = Path(repo).resolve()/'ml/notebooks/12_compact_metadata_ablation.ipynb'
    document, provenance = read_notebook_document(path)
    namespace = {name: globals()[name] for name in
                 ('hashlib','importlib','json','math','os','platform','shutil','tempfile','time',
                  'datetime','timezone','Path','uuid4','ZipFile','ZIP_DEFLATED','lgb','np','pd','uniform_filter','roc_auc_score')}
    namespace['__name__'] = 'notebook12_verified_definitions'
    verified = {}
    for cell_id, names in REUSE_SPECS.items():
        tree = selected_ast(document, cell_id, names)
        digest = hashlib.sha256(ast.dump(tree, include_attributes=False).encode()).hexdigest()
        if digest != REUSE_AST_SHA256[cell_id]:
            raise ValueError('Notebook 12 definitions changed: '+cell_id+'. Pull matching notebooks 12 and 13.')
        exec(compile(tree, str(path)+':'+cell_id, 'exec'), namespace)
        verified[cell_id] = digest
    provenance['definition_ast_sha256'] = verified
    return namespace, provenance


def resolve_source12(cfg):
    if cfg['run_dir']:
        source = Path(cfg['run_dir']).expanduser().resolve()
    else:
        candidates = []
        for path in Path(cfg['output_root']).expanduser().glob('*/run.json'):
            info = json.loads(path.read_text())
            if (info.get('status') == 'complete' and info.get('study') == 'compact-quality'
                    and 'cell' in info.get('config', {}).get('splits', [])
                    and set(cfg['seeds']) <= set(info.get('config', {}).get('seeds', []))):
                candidates.append(path)
        if not candidates:
            raise FileNotFoundError('Set run_dir to a completed notebook 12 directory with all requested seeds and cell splits.')
        source = max(candidates, key=lambda path: path.stat().st_mtime_ns).parent
    prior = json.loads((source/'run.json').read_text())
    if prior.get('status') != 'complete' or prior.get('study') != 'compact-quality':
        raise ValueError('run_dir must be a completed notebook 12 result directory, not 08d, 09, 13 or a ZIP')
    if ('cell' not in prior['config']['splits'] or not set(cfg['seeds']) <= set(prior['config']['seeds'])):
        raise ValueError('The notebook 12 source does not contain all requested outer cell splits')
    required = ('splits.csv','quality_by_setting.csv','test_predictions.csv','models.csv','feature_definitions.json','input_manifest.json')
    for name in required:
        if not (source/name).is_file():
            raise FileNotFoundError('Missing notebook 12 artifact: '+str(source/name))
    print('Using notebook 12 run:', source, flush=True)
    return source, prior

## Freeze the outer test; vary training-only validation

| Policy | Inner validation selection from outer training queries |
|---|---|
| `query` | Every tenth sorted training query ID, exactly matching `gate_eval.fit` |
| `cell` | A deterministic SHA-256 order of training cell IDs, salted by seed; hold out ceil(10% of cells) |

All feature variants share the same inner assignment. The grouped policy requires zero shared cell IDs between fitting
and validation. The selected groups depend only on training IDs, not labels or calibration/test outcomes. Ten percent of
cells need not contain exactly ten percent of queries. Both fitting and validation memberships are exported.

**Validation selects the number of trees. Calibration selects the acceptance cutoff.** These are separate datasets and
decisions. Outer calibration and test IDs are copied from 12, never regenerated. The existing cohort excludes exact counts
below 10, including zero; relative error is `abs(estimate - truth) / truth`.

In [ ]:
def make_inner_assignments(outer_train, seed, policy):
    train = outer_train[['qId','cell']].drop_duplicates().sort_values('qId').reset_index(drop=True)
    if train.qId.duplicated().any() or train.cell.isna().any():
        raise ValueError('Each training query must have exactly one known source-cell ID')
    if policy == 'query':
        validation_ids = set(train.qId.to_numpy()[::10])
    elif policy == 'cell':
        cells = sorted(set(train.cell.astype(str)))
        if len(cells) < 2:
            raise ValueError('At least two training cell groups are required')
        ordered = sorted(cells, key=lambda cell: hashlib.sha256(f'13-inner-cell:{seed}:{cell}'.encode()).digest())
        n = min(len(cells)-1, max(1, math.ceil(INNER_FRACTION*len(cells))))
        held = set(ordered[:n])
        validation_ids = set(train.loc[train.cell.astype(str).isin(held), 'qId'])
    else:
        raise ValueError('Unknown validation policy: '+str(policy))
    train['role'] = np.where(train.qId.isin(validation_ids), 'validation', 'fit')
    if set(train.role) != {'fit','validation'}:
        raise ValueError('Both inner fit and validation must be nonempty')
    return train


def prepare_experiment(cfg, destination, state):
    source, prior = resolve_source12(cfg)
    namespace, reuse = load_reused_definitions(cfg['repo_root'])
    for name, digest in prior['source_code_sha256'].items():
        if file_sha(Path(cfg['repo_root'])/'ml'/name) != digest:
            raise ValueError('Code differs from notebook 12: '+name+'. Use the matching checkout.')
    versions = {name: importlib.metadata.version(name) for name in prior['package_versions']}
    if versions != prior['package_versions']:
        raise ValueError('Package versions differ from notebook 12; use the same Databricks runtime to isolate validation policy: '+str(versions))
    source08d = Path(prior['source_run'])
    if not (source08d/'run.json').is_file():
        raise FileNotFoundError('Notebook 12 source 08d directory is unavailable: '+str(source08d))
    if file_sha(source08d/'run.json') != prior['source_manifest_sha256']:
        raise ValueError('Source 08d manifest changed since notebook 12')
    inherited = dict(prior['config'])
    inherited.update(repo_root=cfg['repo_root'], output_root=cfg['output_root'], run_dir=str(source08d),
                     data_dir=cfg['data_dir'], histogram_dir=cfg['histogram_dir'], seeds=cfg['seeds'], splits=['cell'],
                     merge_rates=','.join(map(str, prior['config']['resolved_base_rates'])))
    preparation = destination/'preparation'; preparation.mkdir()
    tick = time.perf_counter()
    data = namespace['prepare_artifacts'](inherited, preparation, state)
    state['timings']['artifact_and_feature_preparation_seconds'] = time.perf_counter()-tick
    definitions = json.loads((source/'feature_definitions.json').read_text())
    for variant in VARIANTS:
        if data['definitions'][variant] != definitions[variant]:
            raise ValueError('Feature definition differs from notebook 12: '+variant)
    data['definitions'] = {variant: data['definitions'][variant] for variant in VARIANTS}
    data['frames'] = {variant: data['frames'][variant] for variant in VARIANTS}
    if inherited['resolved_base_rates'] != prior['config']['resolved_base_rates']:
        raise ValueError('Resolved physical base rates differ from notebook 12')
    input_identity = json.loads((source/'input_manifest.json').read_text())
    current_identity = json.loads((preparation/'input_manifest.json').read_text())
    if ({key: item['sha256'] for key,item in input_identity.items()} !=
            {key: item['sha256'] for key,item in current_identity.items()}):
        raise ValueError('Input identity differs from notebook 12')
    outer = pd.read_csv(source/'splits.csv')
    outer = outer[(outer.split_kind == 'cell') & outer.seed.isin(cfg['seeds'])].copy()
    if outer.duplicated(['seed','qId']).any():
        raise ValueError('Duplicate outer split identifiers')
    cohort = set(data['frames']['merged_only'].qId)
    for seed in cfg['seeds']:
        part = outer[outer.seed == seed]
        if set(part.qId) != cohort or set(part.split) != {'train','calibration','test'}:
            raise ValueError('Incomplete outer split in notebook 12')
        mapped = data['cells'].loc[part.qId].astype(str).to_numpy()
        if not np.array_equal(mapped, part.cell.astype(str).to_numpy()):
            raise ValueError('Source-cell assignment differs from notebook 12')
        groups = [set(part.loc[part.split == label,'cell']) for label in ('train','calibration','test')]
        if any(groups[i] & groups[j] for i in range(3) for j in range(i)):
            raise ValueError('Outer source-cell groups overlap')
    outer.to_csv(destination/'outer_splits.csv', index=False)
    assignments, inner_summary = [], []
    for seed in cfg['seeds']:
        training = outer[(outer.seed == seed) & (outer.split == 'train')]
        for policy in POLICIES:
            inner = make_inner_assignments(training, seed, policy)
            fit, validation = (inner[inner.role == label] for label in ('fit','validation'))
            overlap = set(fit.cell) & set(validation.cell)
            if policy == 'cell' and overlap:
                raise ValueError('Inner grouped validation overlaps fitting cells')
            assignments.append(inner.assign(seed=seed, validation_policy=policy))
            inner_summary.append(dict(seed=seed, validation_policy=policy, fit_queries=len(fit),
                validation_queries=len(validation), fit_cells=fit.cell.nunique(), validation_cells=validation.cell.nunique(),
                overlapping_cells=len(overlap), validation_query_fraction=len(validation)/len(inner),
                validation_sibling_share=float(validation.cell.isin(set(fit.cell)).mean())))
    inner = pd.concat(assignments, ignore_index=True)
    inner.to_csv(destination/'inner_validation_assignments.csv', index=False)
    pd.DataFrame(inner_summary).to_csv(destination/'inner_split_summary.csv', index=False)
    references = dict(quality=pd.read_csv(source/'quality_by_setting.csv'),
                      predictions=pd.read_csv(source/'test_predictions.csv'), models=pd.read_csv(source/'models.csv'))
    namespace['HELPERS'] = data['gate_helpers']
    state.update(source12_run=str(source), source12_manifest_sha256=file_sha(source/'run.json'),
        source12_artifact_sha256={name:file_sha(source/name) for name in
                                 ('splits.csv','quality_by_setting.csv','test_predictions.csv','models.csv','feature_definitions.json','input_manifest.json')},
        reuse_notebook12=reuse, inherited_training_config={key:inherited[key] for key in
              ('max_trees','num_threads','random_seed','resolved_base_rates')}, package_versions=versions,
        variants=list(VARIANTS), feature_counts={variant:len(data['definitions'][variant]) for variant in VARIANTS},
        inner_validation_fraction=INNER_FRACTION, early_stopping_rounds=EARLY_STOPPING_ROUNDS)
    save_json(destination/'feature_definitions.json', data['definitions'])
    return data, namespace, inherited, outer, inner, references

## Train, reproduce the baseline, and calibrate

The three inputs are unchanged: `merged_only` has 14 source-free features, `compact16` adds two metadata features,
and `full_source` adds seven source features (21 total). Every learned model excludes SA inputs.

Both policies use the inherited LightGBM parameters, seed and `early_stopping(100)`. The query policy calls the existing
fit helper directly. Its test scores, calibrated decisions, counts and cutoffs must reproduce notebook 12 before a policy
comparison is accepted. Model hashes are recorded separately; CPU/runtime versions can matter.

Each model's cutoff is selected using only the same outer calibration queries, with a 25% error limit and a 5% empirical
accepted-violation target. This is not a population guarantee. Test labels do not select trees or gates. Zero accepted
answers have undefined violation, not 0%. Fixed 5%/10% score-only ranking diagnostics retain entire ties and are separate
from deployment gates; a model with few trees may accept much more than the nominal diagnostic fraction.

In [ ]:
def train_and_evaluate(cfg, destination, state, data, namespace, inherited, outer, inner, references):
    quality, fixed, models, predictions, calibration_predictions, audits = [], [], [], [], [], []
    params = dict(data['gate_helpers'].PARAMS, n_estimators=inherited['max_trees'], n_jobs=inherited['num_threads'])
    state['model_parameters'] = params
    for seed in cfg['seeds']:
        outside = outer[outer.seed == seed]
        groups = {name:set(outside.loc[outside.split == name,'qId']) for name in ('train','calibration','test')}
        for policy in POLICIES:
            assignment = inner[(inner.seed == seed) & (inner.validation_policy == policy)]
            fit_ids = set(assignment.loc[assignment.role == 'fit','qId'])
            validation_ids = set(assignment.loc[assignment.role == 'validation','qId'])
            if fit_ids & validation_ids or fit_ids | validation_ids != groups['train']:
                raise ValueError('Inner assignment is not a partition of outer training queries')
            for variant in VARIANTS:
                names, frame = data['definitions'][variant], data['frames'][variant]
                training = frame[frame.qId.isin(groups['train'])]
                fit, validation = (training[training.qId.isin(ids)] for ids in (fit_ids,validation_ids))
                tick = time.perf_counter()
                if policy == 'query':
                    model = data['gate_helpers'].fit(training, names, seed, params=params, stop=EARLY_STOPPING_ROUNDS)
                else:
                    model = lgb.LGBMRegressor(random_state=seed, **params)
                    model.fit(fit[names], fit.label, eval_set=[(validation[names], validation.label)],
                              callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)])
                fit_seconds = time.perf_counter()-tick
                folder = destination/'models'/policy/f'seed_{seed}'/variant; folder.mkdir(parents=True)
                model.booster_.save_model(str(folder/'model.txt'))
                schema = dict(features=names, outer_split='cell', seed=seed, validation_policy=policy,
                    variant=variant, label='log(max(relative_error,0.001))', min_exact=10,
                    params=params, early_stopping_rounds=EARLY_STOPPING_ROUNDS, refit_on_outer_train=False)
                save_json(folder/'schema.json',schema)
                model_digest = file_sha(folder/'model.txt')
                losses = model.best_score_['valid_0']
                models.append(dict(seed=seed, validation_policy=policy, variant=variant, fit_seconds=fit_seconds,
                    best_iteration=int(model.best_iteration_), tree_count=model.booster_.num_trees(),
                    validation_loss_metric=next(iter(losses)), validation_loss=float(next(iter(losses.values()))),
                    fit_query_histogram_rows=len(fit),validation_query_histogram_rows=len(validation),
                    model_bytes=(folder/'model.txt').stat().st_size, model_sha256=model_digest))
                for base in data['base_names']:
                    part = frame[frame.histogram_key == base]
                    cal, test = (part[part.qId.isin(groups[label])].sort_values('qId') for label in ('calibration','test'))
                    tick = time.perf_counter()
                    cs = model.predict(cal[names], num_threads=inherited['num_threads'])
                    ts = model.predict(test[names], num_threads=inherited['num_threads'])
                    batch_seconds = time.perf_counter()-tick
                    if not np.isfinite(cs).all() or not np.isfinite(ts).all():
                        raise ValueError('Nonfinite model prediction')
                    evaluation = namespace['evaluate_score'](cal,test,cs,ts,epsilon=.25,delta=.05)
                    prefix = dict(split_kind='cell',seed=seed,validation_policy=policy,variant=variant,
                                  base_rate=data['rates'][base],physical_rate=data['rates'][base])
                    quality.append(dict(**prefix, **evaluation, score='learned',
                        score_space='predicted log-relative-error',calibration_count=len(cal),batch_prediction_seconds=batch_seconds))
                    fixed.extend(dict(**prefix,**item,score='learned') for item in namespace['fixed_coverage'](test,ts))
                    cutoff = -np.inf if evaluation['accept_none'] else evaluation['cutoff']
                    prediction = test[['qId','histogram_key','estimate','truth','relErr']].assign(
                        score_log=ts,accepted=ts<=cutoff,**prefix)
                    predictions.append(prediction)
                    calibration_predictions.append(cal[['qId','histogram_key','estimate','truth','relErr']].assign(
                        score_log=cs,accepted=cs<=cutoff,**prefix))
                    if policy == 'query':
                        old = references['predictions']
                        old = old[(old.split_kind == 'cell') & (old.seed == seed) & (old.variant == variant)
                                  & (old.base_rate == data['rates'][base])].sort_values('qId')
                        old_quality = references['quality']
                        old_quality = old_quality[(old_quality.split_kind == 'cell') & (old_quality.seed == seed)
                            & (old_quality.variant == variant) & (old_quality.base_rate == data['rates'][base])
                            & (old_quality.score == 'learned')]
                        old_model = references['models']
                        old_model = old_model[(old_model.split_kind == 'cell') & (old_model.seed == seed) & (old_model.variant == variant)]
                        if (len(old) != len(test) or len(old_quality) != 1 or len(old_model) != 1
                                or not np.array_equal(old.qId.to_numpy(),test.qId.to_numpy())):
                            raise ValueError('Incomplete notebook 12 baseline reference')
                        max_gap = float(np.max(np.abs(ts-old.score_log.to_numpy())))
                        score_match = bool(np.allclose(ts,old.score_log.to_numpy(),rtol=1e-9,atol=1e-10))
                        decision_match = bool(np.array_equal(ts<=cutoff,old.accepted.to_numpy()))
                        metric_match = (evaluation['accepted_count'] == old_quality.iloc[0].accepted_count and
                                        evaluation['violation_count'] == old_quality.iloc[0].violation_count)
                        cutoff_match = (evaluation['accept_none'] == bool(old_quality.iloc[0].accept_none) and
                            (evaluation['accept_none'] or np.isclose(evaluation['cutoff'],old_quality.iloc[0].cutoff,rtol=1e-10,atol=1e-12)))
                        audits.append(dict(seed=seed,variant=variant,base_rate=data['rates'][base],max_score_gap=max_gap,
                            scores_match=score_match,decisions_match=decision_match,counts_match=bool(metric_match),
                            cutoff_match=bool(cutoff_match),model_sha256_match=model_digest == old_model.iloc[0].model_sha256))
                        pd.DataFrame(audits).to_csv(destination/'baseline_reproduction.csv',index=False)
                        if not (score_match and decision_match and metric_match and cutoff_match):
                            raise ValueError('Query-validation baseline does not reproduce notebook 12; inspect baseline_reproduction.csv before interpreting a policy change')
                print('seed',seed,'validation',policy,variant,'trees',model.booster_.num_trees(),
                      'fit seconds',round(fit_seconds,2),flush=True)
    quality = pd.DataFrame(quality); fixed = pd.DataFrame(fixed)
    quality.to_csv(destination/'quality_by_setting.csv',index=False)
    fixed.to_csv(destination/'fixed_coverage.csv',index=False)
    pd.DataFrame(models).to_csv(destination/'models.csv',index=False)
    pd.concat(predictions,ignore_index=True).to_csv(destination/'test_predictions.csv',index=False)
    pd.concat(calibration_predictions,ignore_index=True).to_csv(destination/'calibration_predictions.csv',index=False)
    return quality, fixed

## Save a reproducible policy comparison

Compare each candidate with its matching seed, feature variant and base histogram. Read accepted good-answer counts,
violations, acceptance and AUC together. Aggregate cases across seeds/rates are descriptive and are not independent trials.

This experiment reuses an inspected test set. It can diagnose the validation-policy hypothesis, but it cannot establish
a final improvement claim. Source-cell IDs are inferred generating groups, not strict non-overlapping geographic regions.
Any settings selected after inspecting these results require fresh untouched grouped queries for final validation.

In [ ]:
def export_results(destination):
    with ZipFile(destination/'group_validation_results.zip','w',ZIP_DEFLATED) as archive:
        for path in destination.rglob('*'):
            if (path.is_file() and path.suffix in {'.csv','.json','.md','.png'}
                    and 'models' not in path.relative_to(destination).parts):
                archive.write(path,path.relative_to(destination))


def run_group_validation(cfg):
    global lgb,uniform_filter,roc_auc_score
    import lightgbm as lgb
    from scipy.ndimage import uniform_filter
    from sklearn.metrics import roc_auc_score
    destination = Path(cfg['output_root']).expanduser().resolve()/('group-validation-'+uuid4().hex[:12])
    destination.mkdir(parents=True,exist_ok=False)
    state = dict(status='running',study='group-validation',config=cfg,timings={},
        started_utc=datetime.now(timezone.utc).isoformat(),python=platform.python_version(),epsilon=.25,delta=.05,
        notes=['This is an exploratory validation-policy comparison, not a new generalization claim.',
               'Outer cell train/calibration/test IDs and input/features/LightGBM settings are inherited from notebook 12.',
               'Both policies retrain; original query validation is checked against saved notebook 12 predictions and gates.',
               'Cell validation also changes the fitting observations; realized query/cell counts are reported.',
               'No refit on the full outer training set is performed after early stopping, matching the existing workflow.',
               'Group assignment uses training IDs only, never calibration/test labels or outcomes.',
               'Calibration independently chooses each model gate with the same 25% error limit and 5% empirical risk target.',
               'The reused preparation also builds auxiliary notebook 12 representations; only three variants are trained here.',
               'No runtime microbenchmark, E2E speedup, exact fallback or storage improvement is evaluated here.',
               'Inferred source-cell groups are not strict geographic separation or a new dataset.',
               'Seeds and merge rates can repeat queries; descriptive cases are not independent trials.'])
    save_json(destination/'run.json',state); started = time.perf_counter()
    try:
        data,namespace,inherited,outer,inner,references = prepare_experiment(cfg,destination,state)
        save_json(destination/'run.json',state)
        tick = time.perf_counter()
        quality,fixed = train_and_evaluate(cfg,destination,state,data,namespace,inherited,outer,inner,references)
        state['timings']['training_calibration_evaluation_seconds'] = time.perf_counter()-tick
        keys = ['seed','variant','base_rate']
        joined = quality[quality.validation_policy == 'cell'].merge(
            quality[quality.validation_policy == 'query'],on=keys,suffixes=('_cell','_query'),validate='one_to_one')
        pairs = joined[keys].copy()
        for metric in ('acceptance','violation','good_accepted_count','good_recall','auc'):
            pairs[metric+'_query'] = joined[metric+'_query']
            pairs[metric+'_cell'] = joined[metric+'_cell']
            pairs[metric+'_delta'] = joined[metric+'_cell']-joined[metric+'_query']
        pairs.to_csv(destination/'paired_deltas.csv',index=False)
        summary = quality.groupby(['validation_policy','variant'],as_index=False).agg(
            settings=('seed','size'),acceptance_mean=('acceptance','mean'),acceptance_min=('acceptance','min'),
            acceptance_max=('acceptance','max'),violation_mean_defined=('violation','mean'),
            zero_acceptance_settings=('accepted_count',lambda x:int((x==0).sum())),
            settings_above_target=('violation',lambda x:int((x>.05).sum())),
            accepted_cases=('accepted_count','sum'),violation_cases=('violation_count','sum'),
            good_accepted_cases=('good_accepted_count','sum'),auc_mean=('auc','mean'))
        summary['pooled_case_violation'] = np.where(summary.accepted_cases>0,
            summary.violation_cases/summary.accepted_cases,np.nan)
        summary.to_csv(destination/'summary.csv',index=False)
        fixed.groupby(['validation_policy','variant','nominal_coverage'],as_index=False).agg(
            settings=('seed','size'),violation_mean=('violation','mean'),actual_coverage_mean=('actual_coverage','mean'),
            actual_coverage_min=('actual_coverage','min'),actual_coverage_max=('actual_coverage','max'),
            tie_extra_cases=('tie_extra_count','sum')).to_csv(destination/'fixed_coverage_summary.csv',index=False)
        (destination/'analysis_en.md').write_text(
            '# Group-aware validation experiment\n\n'
            'Compare query and cell validation within the SAME outer cell split, seed, base rate and feature variant. '
            'Use quality_by_setting.csv and paired_deltas.csv; acceptance gains must be read with accepted violations '
            'and good-answer counts. Zero acceptance has undefined risk. Descriptive means can hide individual failures.\n\n'
            'baseline_reproduction.csv checks the retrained query-validation policy against notebook 12. '
            'inner_validation_assignments.csv and inner_split_summary.csv expose fitting/validation IDs, group overlap '
            'and realized fractions. Cell validation changes which observations fit the model; this is a policy comparison, '
            'not a controlled comparison with identical fitting observations. Neither policy refits on all outer train rows.\n\n'
            'Validation selects the early-stopping iteration. The separate outer calibration set selects the gate. '
            'Fixed 5%/10% coverage diagnostics select using test scores only and retain complete ties; actual coverage '
            'can be much larger for a low-tree model. These diagnostic thresholds are not deployment gates.\n\n'
            'The 08d cohort excludes exact counts below 10, including zero. All learned variants exclude SA inputs. '
            'No new features or model families are introduced. Reused notebook 12 preparation costs are setup costs, '
            'not query latency. This experiment makes no runtime, storage or E2E claim.\n\n'
            'The test has already been inspected. Any apparent improvement remains exploratory and requires fresh '
            'untouched grouped queries before a final claim. Source-cell groups are inferred from query size and centre; '
            'they are not strict non-overlapping geographic regions. Repeated seed/rate cases are not independent trials.\n')
        state.update(status='complete',wall_seconds=time.perf_counter()-started,
                     finished_utc=datetime.now(timezone.utc).isoformat())
        save_json(destination/'run.json',state); export_results(destination)
        print('COMPLETE:',destination,flush=True)
        return destination,quality,summary
    except BaseException as error:
        state.update(status='failed',error=f'{type(error).__name__}: {error}',wall_seconds=time.perf_counter()-started)
        save_json(destination/'run.json',state)
        print('FAILED; diagnostics preserved:',destination,flush=True)
        raise

## Execute

The default `report` mode skips this training cell and proceeds to the final report. In `train` mode, both policies run.
If preparation or baseline reproduction fails, the failed run's `run.json` and available audit CSVs are preserved.
Fix the mismatch before interpreting the comparison; there is no automatic alternative model or threshold.

In [ ]:
# RUN_GROUP_VALIDATION
config = load_settings()
_trained_in_this_execution = False
if config['execution_mode']=='train':
    result_dir,quality_by_setting,summary = run_group_validation(config)
    _trained_in_this_execution = True
    display(summary) if 'display' in globals() else print(summary.to_string(index=False))
else:
    print('Report mode: training skipped. The final cell reads an existing completed notebook 13 run.')

## Inspect and share

- `quality_by_setting.csv`, `paired_deltas.csv`: frozen-gate quality and cell-minus-query policy differences.
- `fixed_coverage.csv`: score-ranking diagnostics, including actual coverage and extra tied cases.
- `inner_validation_assignments.csv`, `inner_split_summary.csv`, `outer_splits.csv`: exact memberships and group overlap.
- `baseline_reproduction.csv`: reproduction of notebook 12's query-validation policy on outer cell tests.
- `models.csv`: tree count, validation loss, fitting time, model bytes and hashes.
- `calibration_predictions.csv`, `test_predictions.csv`, `feature_definitions.json`, `run.json`: reproducibility records.

Training writes **`group_validation_results.zip`**. The final full-query report is the main result to share. This training
archive contains CSVs, English notes, provenance and figures, while
model weights and histogram payloads stay in the result directory. The two panels distinguish frozen calibration gates
from a nominal 10% ranking diagnostic; complete ties can change the latter's actual selection fraction.

In [ ]:
# PLOT_GROUP_VALIDATION
if str(get_config('execution_mode')).strip()=='train' and globals().get('_trained_in_this_execution',False):
    import matplotlib.pyplot as plt
    fixed = pd.read_csv(Path(result_dir)/'fixed_coverage.csv')
    fig,axes = plt.subplots(1,2,figsize=(12,4))
    colors = dict(merged_only='tab:blue',compact16='tab:orange',full_source='tab:green')
    for (policy,variant),part in quality_by_setting.groupby(['validation_policy','variant']):
        marker = 'o' if policy=='query' else '^'
        label = f'{variant} / {policy} validation'
        axes[0].scatter(part.acceptance*100,part.violation*100,label=label,marker=marker,color=colors[variant],alpha=.8)
        subset = fixed[(fixed.validation_policy==policy)&(fixed.variant==variant)&(fixed.nominal_coverage==.1)]
        axes[1].scatter(subset.actual_coverage*100,subset.violation*100,label=label,marker=marker,color=colors[variant],alpha=.8)
    for axis in axes:
        axis.axhline(5,color='black',linestyle='--',linewidth=1)
        axis.grid(alpha=.2)
        axis.set(xlabel='Accepted queries (%)',ylabel='Violation among accepted (%)')
    axes[0].set_title('Frozen calibration gates on outer cell test')
    axes[1].set_title('Nominal 10% ranking diagnostic: complete ties')
    axes[0].legend(fontsize=7,bbox_to_anchor=(0,-.2),loc='upper left',ncol=2)
    fig.tight_layout()
    fig.savefig(Path(result_dir)/'validation_policy_quality.png',dpi=160,bbox_inches='tight')
    plt.show()
    export_results(Path(result_dir))
    print('Upload:',Path(result_dir)/'group_validation_results.zip')
else:
    print('Report mode: old training plots skipped. See the final full-query report cell.')

## Main report · All queries, SA comparison and complete risk–coverage curves

**Run the next cell by itself to report an existing completed 13 run. No training cells or model packages are required.**
Leave `report_run_dir` blank to choose the latest completed 13 directory under `output_root`; no stale session `result_dir`
is reused. Set it explicitly to reproduce a particular run. The original `run_dir` widget belongs only to training.

Numerical metrics compare `exp(score_log)` with the actual relative error for every query, including rejected answers.
MAE is reported in percentage points: a prediction of 15% versus actual error 20% differs by 5 percentage points.
No predictions are clipped at 100%. Log-space metrics also match the original training target. These are point predictions,
not confidence bounds. Raw SA scores have no compatible numerical error units; SA is compared using ranking and decisions.
A constant numerical reference predicts the median true error from the matching calibration queries, and its MAE is
evaluated on all test queries. It gives ML's numerical MAE context without treating SA as a percentage prediction.

- `all_query_metrics.csv`, `metrics_summary.csv`: full-query numerical prediction, bias/underestimation, AUC and Spearman.
- `risk_coverage.csv`: ALL complete-tie score thresholds. Test thresholds are descriptive, never new deployment gates.
  The figure includes full and low-coverage views; crosses mark the original calibration-selected gates.
- `gate_results.csv`, `gate_audit.csv`: original ML gates checked against saved calibration; SA uses the same calibration rule.
- `test_scores.csv`, `calibration_scores.csv`, `outer_splits.csv`: the complete cohorts used, with source identity recorded.
- `models.csv`: optional model size/fitting data; these do not establish query latency or E2E efficiency versus SA.

Every report writes a new directory. Zero acceptance has undefined violation, SA is counted once per seed/rate, and repeated
queries across seeds are not independent trials. The original 25%/5% settings are unchanged; existing test reuse is exploratory.
Upload **`full_query_report.zip`**. Missing SA inputs are listed in `missing_inputs.csv`; no incomplete subset is substituted.

In [ ]:
# Standalone full-query report. This cell never imports a training library or fits a model.
import hashlib
import json
import math
import os
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4
from zipfile import ZipFile, ZIP_DEFLATED

import numpy as np
import pandas as pd

REPORT_EPSILON = .25
REPORT_DELTA = .05
REPORT_KEYS = ['seed','validation_policy','variant','base_rate']


def report_defaults():
    try:
        context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        output = '/Workspace/Users/'+context.userName().get()+'/synopsislake-results'
    except (NameError,AttributeError):
        import tempfile
        output = str(Path(tempfile.gettempdir())/'synopsislake-results')
    return dict(output_root=output,report_run_dir='',rows_file='')


def load_report_settings():
    cfg = report_defaults()
    try:
        for key,value in cfg.items():
            try:
                cfg[key] = dbutils.widgets.get(key).strip()
            except Exception:
                dbutils.widgets.text(key,value)
    except NameError:
        cfg = {key:os.environ.get(key.upper(),value).strip() for key,value in cfg.items()}
    return cfg


def report_sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(1024*1024),b''):
            digest.update(block)
    return digest.hexdigest()


def report_json(path,value):
    path = Path(path);temporary=path.with_suffix(path.suffix+'.tmp')
    temporary.write_text(json.dumps(value,indent=2,allow_nan=False,default=str))
    temporary.replace(path)


def report_bool(series):
    result=series.map({True:True,False:False,'true':True,'false':False,'True':True,'False':False})
    if result.isna().any():raise ValueError('Invalid Boolean values in '+str(series.name))
    return result.astype(bool)


def report_cutoff(scores,errors):
    scores,errors=np.asarray(scores,float),np.asarray(errors,float)
    if not len(scores) or not np.isfinite(scores).all() or not np.isfinite(errors).all():
        raise ValueError('Calibration scores and errors must be nonempty and finite')
    order=np.argsort(scores,kind='stable');ordered=scores[order]
    risk=np.cumsum(errors[order]>REPORT_EPSILON)/np.arange(1,len(scores)+1)
    valid=np.flatnonzero((risk<=REPORT_DELTA)&np.r_[ordered[1:]!=ordered[:-1],True])
    return float(ordered[valid[-1]]) if len(valid) else -np.inf


def report_metrics(errors,scores,predicted_error=None):
    errors,scores=np.asarray(errors,float),np.asarray(scores,float)
    if not len(errors) or len(errors)!=len(scores) or not np.isfinite(errors).all() or not np.isfinite(scores).all() or (errors<0).any():
        raise ValueError('Full-query errors/scores must be finite, nonnegative-error vectors of equal length')
    bad=errors>REPORT_EPSILON;nb=int(bad.sum());ng=len(bad)-nb
    ranks=pd.Series(scores).rank(method='average').to_numpy()
    auc=(float(ranks[bad].sum())-nb*(nb+1)/2)/(nb*ng) if nb and ng else np.nan
    truth_ranks=pd.Series(errors).rank(method='average').to_numpy()
    spearman=float(np.corrcoef(ranks,truth_ranks)[0,1]) if np.std(ranks)>0 and np.std(truth_ranks)>0 else np.nan
    result=dict(test_count=len(errors),true_good_count=ng,true_bad_count=nb,
        true_histogram_good_rate=ng/len(errors),auc_bad=auc,spearman_error_score=spearman,
        mae_relative=np.nan,mae_pp=np.nan,median_ae_pp=np.nan,p90_ae_pp=np.nan,rmse_pp=np.nan,bias_pp=np.nan,
        underprediction_rate=np.nan,mean_underestimation_on_bad_pp=np.nan,log_mae=np.nan,log_rmse=np.nan)
    if predicted_error is not None:
        prediction=np.asarray(predicted_error,float)
        if prediction.shape!=errors.shape or not np.isfinite(prediction).all() or (prediction<0).any():
            raise ValueError('Relative-error predictions must be finite and nonnegative; do not clip invalid scores')
        residual=prediction-errors;absolute=np.abs(residual)
        # ML scores are the saved log-error predictions; preserve them even if exp underflows.
        log_residual=scores-np.log(np.maximum(errors,.001))
        result.update(mae_relative=float(absolute.mean()),mae_pp=float(100*absolute.mean()),
            median_ae_pp=float(100*np.median(absolute)),p90_ae_pp=float(100*np.quantile(absolute,.9)),
            rmse_pp=float(100*np.sqrt(np.mean(residual**2))),bias_pp=float(100*residual.mean()),
            underprediction_rate=float((prediction<errors).mean()),
            mean_underestimation_on_bad_pp=float(100*np.maximum(-residual[bad],0).mean()) if nb else np.nan,
            log_mae=float(np.abs(log_residual).mean()),log_rmse=float(np.sqrt(np.mean(log_residual**2))))
    return result


def report_curve(errors,scores):
    errors,scores=np.asarray(errors,float),np.asarray(scores,float)
    report_metrics(errors,scores)
    order=np.argsort(scores,kind='stable');s=scores[order]
    bad=np.cumsum(errors[order]>REPORT_EPSILON)
    ends=np.flatnonzero(np.r_[s[1:]!=s[:-1],True]);n=ends+1
    rows=pd.DataFrame(dict(score_cutoff=s[ends],accepted_count=n,violation_count=bad[ends],
        acceptance=n/len(scores),violation=bad[ends]/n))
    zero=pd.DataFrame([dict(score_cutoff=np.nan,accepted_count=0,violation_count=0,acceptance=0.,violation=np.nan)])
    return pd.concat([zero,rows],ignore_index=True)


def report_gate(errors,scores,cutoff):
    errors,scores=np.asarray(errors,float),np.asarray(scores,float)
    accepted=scores<=cutoff;good=errors<=REPORT_EPSILON
    n=int(accepted.sum());bad=int((accepted&~good).sum())
    return dict(test_count=len(errors),accepted_count=n,violation_count=bad,good_accepted_count=n-bad,
        good_rejected_count=int((~accepted&good).sum()),bad_rejected_count=int((~accepted&~good).sum()),
        true_good_count=int(good.sum()),true_bad_count=int((~good).sum()),acceptance=n/len(errors),
        violation=bad/n if n else np.nan,accepted_good_rate=(n-bad)/n if n else np.nan,
        good_answer_recall=(n-bad)/int(good.sum()) if good.any() else np.nan,
        bad_answer_detection_recall=int((~accepted&~good).sum())/int((~good).sum()) if (~good).any() else np.nan,
        cutoff=cutoff if math.isfinite(cutoff) else np.nan,accept_none=not math.isfinite(cutoff))


def report_resolve_source(cfg):
    if cfg['report_run_dir']:
        source=Path(cfg['report_run_dir']).expanduser().resolve()
    else:
        candidates=[]
        for path in Path(cfg['output_root']).expanduser().glob('*/run.json'):
            state=json.loads(path.read_text())
            if state.get('status')=='complete' and state.get('study')=='group-validation':candidates.append(path)
        if not candidates:
            raise FileNotFoundError('Set report_run_dir to a completed notebook 13 result directory; report mode does not train models.')
        source=max(candidates,key=lambda path:path.stat().st_mtime_ns).parent
    prior=json.loads((source/'run.json').read_text())
    if prior.get('status')!='complete' or prior.get('study')!='group-validation':
        raise ValueError('report_run_dir must point to a completed notebook 13 result, not notebook 12 or a ZIP')
    if prior.get('epsilon')!=REPORT_EPSILON or prior.get('delta')!=REPORT_DELTA:
        raise ValueError('The original 25% error limit and 5% calibration target must be unchanged')
    print('Full-query report source:',source,flush=True)
    return source,prior


def report_read_predictions(source):
    names=['test_predictions.csv','calibration_predictions.csv','quality_by_setting.csv','outer_splits.csv']
    identity={name:report_sha(source/name) for name in names}
    test=pd.read_csv(source/'test_predictions.csv');cal=pd.read_csv(source/'calibration_predictions.csv')
    quality=pd.read_csv(source/'quality_by_setting.csv');outer=pd.read_csv(source/'outer_splits.csv')
    for frame in (test,cal):
        if frame.duplicated(REPORT_KEYS+['qId']).any():raise ValueError('Duplicate prediction keys')
        frame['accepted']=report_bool(frame.accepted)
        if not np.isfinite(frame[['truth','estimate','relErr','score_log','base_rate','qId']].to_numpy(float)).all():
            raise ValueError('Saved prediction values must be finite')
        if (frame.truth<10).any() or (frame.estimate<0).any() or (frame.relErr<0).any():
            raise ValueError('Saved predictions violate the original exact-count >=10 cohort')
        if not np.allclose(frame.relErr,abs(frame.estimate-frame.truth)/frame.truth,rtol=1e-12,atol=1e-12):
            raise ValueError('Saved true relative errors differ from the original answers')
    if quality.duplicated(REPORT_KEYS).any() or outer.duplicated(['seed','qId']).any():
        raise ValueError('Duplicate quality or outer split keys')
    quality['accept_none']=report_bool(quality.accept_none)
    settings=lambda frame:set(map(tuple,frame[REPORT_KEYS].drop_duplicates().to_numpy()))
    if settings(test)!=settings(cal) or settings(test)!=settings(quality):
        raise ValueError('Saved calibration/test/quality settings differ')
    for seed,part in outer.groupby('seed'):
        if set(part.split)!={'train','calibration','test'}:raise ValueError('Incomplete outer split')
        cells=[set(part.loc[part.split==role,'cell']) for role in ('train','calibration','test')]
        if any(cells[i]&cells[j] for i in range(3) for j in range(i)):
            raise ValueError('Outer source-cell groups overlap')
    for frame,role in ((test,'test'),(cal,'calibration')):
        for key,part in frame.groupby(REPORT_KEYS):
            ids=set(outer.loc[(outer.seed==key[0])&(outer.split==role),'qId'])
            if set(part.qId)!=ids:raise ValueError('Prediction cohort differs from the frozen outer '+role+' split')
    # One physical answer per query/rate, regardless of seed, feature variant or validation policy.
    physical=pd.concat([test,cal],ignore_index=True)
    for column in ('truth','estimate','relErr'):
        bounds=physical.groupby(['qId','base_rate'])[column].agg(['min','max'])
        if not np.allclose(bounds['min'],bounds['max'],rtol=1e-12,atol=1e-12):
            raise ValueError('Physical answers differ across models: '+column)
    for name,digest in identity.items():
        if report_sha(source/name)!=digest:raise ValueError('Source output changed while reading: '+name)
    return test,cal,quality,outer,identity


def report_sa_scores(source,cfg,test,cal,missing,state):
    path=source/'preparation/input_manifest.json'
    if not path.is_file():
        missing.append(dict(input='preparation/input_manifest.json',path=str(path),reason='Cannot verify the original SA logger'))
        return None,None
    manifest=json.loads(path.read_text());state['input_manifest_sha256']=report_sha(path)
    if 'run/rows.csv' not in manifest:
        missing.append(dict(input='run/rows.csv',path='',reason='SA logger not listed in saved input manifest'))
        return None,None
    record=manifest['run/rows.csv']
    candidates=[Path(cfg['rows_file'])] if cfg['rows_file'] else [Path(record['path'])]
    logger=next((p.expanduser().resolve() for p in candidates if p.expanduser().is_file()),None)
    if logger is None:
        missing.append(dict(input='run/rows.csv',path='; '.join(map(str,candidates)),reason='Set rows_file to the matching original logger for a full SA comparison'))
        return None,None
    if report_sha(logger)!=record['sha256']:raise ValueError('SA logger hash changed since notebook 13')
    required={'method','mergeRate','qId','gt','est','predErr'}
    header=set(pd.read_csv(logger,nrows=0).columns)
    if not required<=header:
        missing.append(dict(input='SA predErr',path=str(logger),reason='Missing columns: '+','.join(sorted(required-header))))
        return None,None
    logged=pd.read_csv(logger,usecols=list(required));test_parts=[];cal_parts=[]
    for frame,parts in ((test,test_parts),(cal,cal_parts)):
        for (seed,rate),group in frame.groupby(['seed','base_rate'],sort=True):
            canonical=group.sort_values(['qId','variant','validation_policy']).drop_duplicates('qId')
            old=logged[(logged.method=='ARM')&np.isclose(logged.mergeRate,rate,rtol=0,atol=1e-12)]
            if old.qId.duplicated().any():raise ValueError('Duplicate SA logger query IDs')
            joined=canonical[['qId','truth','estimate','relErr']].merge(old[['qId','gt','est','predErr']],
                on='qId',how='left',validate='one_to_one',indicator=True)
            matched=joined['_merge'].eq('both')
            if (matched&(~np.isfinite(joined['gt'])|~np.isfinite(joined.est))).any():
                raise ValueError('Matched SA logger rows contain invalid truth/estimates')
            if (matched&(~np.isclose(joined['gt'],joined.truth,rtol=0,atol=1e-9)|((joined.est-joined.estimate).abs()>.001))).any():
                raise ValueError('SA logger does not match notebook 13 truths/histogram estimates')
            if not matched.all() or not np.isfinite(joined.predErr).all():
                missing.append(dict(input=f'SA seed {seed} rate {rate}',path=str(logger),
                    reason='Incomplete/nonfinite SA cohort; no partial-subset comparison is reported'))
                return None,None
            parts.append(joined[['qId','truth','estimate','relErr']].assign(seed=seed,base_rate=rate,
                validation_policy='not_applicable',variant='sa',score=joined.predErr.to_numpy(),
                score_space='raw SA heuristic; higher means predicted worse',predicted_error=np.nan))
    if report_sha(logger)!=record['sha256']:raise ValueError('SA logger changed while reading')
    state['sa_logger']=dict(path=str(logger),sha256=record['sha256'])
    return pd.concat(test_parts,ignore_index=True),pd.concat(cal_parts,ignore_index=True)


def report_make_figures(destination,test_scores,curves,metrics,gates=None):
    import matplotlib.pyplot as plt
    primary='cell' if 'cell' in set(test_scores.validation_policy) else sorted(set(test_scores.validation_policy)-{'not_applicable'})[0]
    learned=test_scores[(test_scores.variant!='sa')&(test_scores.validation_policy==primary)]
    variants=sorted(learned.variant.unique());sample_rows=[]
    fig,axes=plt.subplots(1,len(variants),figsize=(5*len(variants),4),squeeze=False)
    for ax,variant in zip(axes[0],variants):
        part=learned[learned.variant==variant].copy()
        part['plot_order']=[hashlib.sha256(f'{s}:{q}:{r:.17g}'.encode()).hexdigest()
            for s,q,r in part[['seed','qId','base_rate']].itertuples(index=False,name=None)]
        sample=part.sort_values('plot_order').head(4000)
        ax.scatter(sample.relErr,sample.predicted_error,s=4,alpha=.2)
        maximum=max(float(sample.relErr.max()),float(sample.predicted_error.max()),.25)
        ax.plot([0,maximum],[0,maximum],color='black',linestyle='--')
        ax.set(xscale='symlog',yscale='symlog',xlabel='True histogram relative error',ylabel='ML-predicted relative error',title=variant)
        sample_rows.append(dict(variant=variant,validation_policy=primary,metric_rows=len(part),display_rows=len(sample)))
    fig.suptitle('All queries scored; deterministic display sample; seeds may repeat queries')
    fig.tight_layout();fig.savefig(destination/'error_prediction.png',dpi=150);plt.show();plt.close(fig)
    pd.DataFrame(sample_rows).to_csv(destination/'plot_sample_counts.csv',index=False)
    color={'merged_only':'tab:blue','compact16':'tab:orange','full_source':'tab:green','sa':'black'}
    curve_groups=list(curves.groupby(['seed','base_rate'],sort=True))
    fig,axes=plt.subplots(2,len(curve_groups),figsize=(5*len(curve_groups),7),squeeze=False)
    for column,((seed,rate),part) in enumerate(curve_groups):
        selected=part[part.validation_policy.isin([primary,'not_applicable'])]
        for row,limit in enumerate((100,20)):
            ax=axes[row,column]
            for variant,group in selected.groupby('variant'):
                ax.plot(group.acceptance*100,group.violation*100,label=variant,color=color.get(variant),linewidth=1,marker='.',markersize=1.2)
            if gates is not None:
                frozen=gates[(gates.seed==seed)&(gates.base_rate==rate)&gates.validation_policy.isin([primary,'not_applicable'])]
                for gate in frozen.itertuples():
                    ax.scatter(gate.acceptance*100,gate.violation*100,marker='x',s=40,color=color.get(gate.variant),zorder=4)
            ax.axhline(5,color='gray',linestyle='--',label='Original 5% target')
            ax.set(xlim=(0,limit),ylim=(0,100),xlabel='Accepted queries (%)',ylabel='Violation among accepted (%)',
                   title=f'Seed {seed}, rate {rate:.5f}'+(' / low coverage' if row else ''))
            ax.grid(alpha=.2)
    axes[0,0].legend(fontsize=7);fig.suptitle(f'Complete risk–coverage / {primary} validation; x = frozen calibration gate; ties stay together')
    fig.tight_layout();fig.savefig(destination/'risk_coverage.png',dpi=150);plt.show();plt.close(fig)
    selected=metrics[metrics.validation_policy.isin([primary,'not_applicable'])]
    fig,axes=plt.subplots(1,2,figsize=(10,4))
    names=sorted(selected.variant.unique())
    for i,variant in enumerate(names):
        part=selected[selected.variant==variant]
        axes[0].scatter(np.full(len(part),i),part.auc_bad,label=variant,color=color.get(variant))
        if variant!='sa':axes[1].scatter(np.full(len(part),i),part.mae_pp,color=color.get(variant))
    axes[0].set(ylabel='AUC for error >25%',title='Every test query: one point per seed/rate')
    axes[1].set(ylabel='Error-prediction MAE (percentage points)',title='ML numerical error; SA raw score is not comparable')
    for ax in axes:
        ax.set_xticks(range(len(names)),names,rotation=20);ax.grid(alpha=.2)
    fig.tight_layout();fig.savefig(destination/'all_query_metrics.png',dpi=150);plt.show();plt.close(fig)


def report_export(destination):
    with ZipFile(destination/'full_query_report.zip','w',ZIP_DEFLATED) as archive:
        for path in sorted(destination.rglob('*')):
            if path.is_file() and path.suffix in {'.csv','.json','.md','.png'}:archive.write(path,path.relative_to(destination))


def run_full_report(cfg):
    destination=Path(cfg['output_root']).expanduser().resolve()/('full-query-report-'+uuid4().hex[:12])
    destination.mkdir(parents=True,exist_ok=False)
    state=dict(status='running',study='full-query-report',config=cfg,started_utc=datetime.now(timezone.utc).isoformat(),
        epsilon=REPORT_EPSILON,delta=REPORT_DELTA,package_versions=dict(numpy=np.__version__,pandas=pd.__version__),
        notes=['No training, model loading or threshold retuning.',
               'Numerical ML metrics use exp(score_log) without clipping, against the actual unfloored relative error.',
               'The log-trained point estimate is not an unbiased conditional mean or an error/confidence bound.',
               'Raw SA scores have no compatible relative-error units; their numerical prediction MAE is undefined.',
               'All-query metrics use every test row; only scatter displays are sampled deterministically.',
               'Complete score ties remain together in risk–coverage curves. Zero acceptance has undefined risk.',
               'SA is reported once per seed/rate, independent of ML validation policy.',
               'Risk–coverage test thresholds are diagnostic, not deployment gates.',
               'This inspected-test report is exploratory; repeated seed/rate cases are not independent observations.',
               'Model fit time and bytes are not SA query latency or E2E performance. No comparable runtime measurements are available for these new models.'])
    report_json(destination/'run.json',state)
    try:
        source,prior=report_resolve_source(cfg)
        test,cal,quality,outer,identity=report_read_predictions(source)
        state.update(source13_run=str(source),source13_manifest_sha256=report_sha(source/'run.json'),source_artifact_sha256=identity)
        metrics,gates,audits,curves,test_parts,cal_parts=[],[],[],[],[],[]
        quality_index=quality.set_index(REPORT_KEYS)
        cal_index={key:part for key,part in cal.groupby(REPORT_KEYS,sort=True)}
        for key,part in test.groupby(REPORT_KEYS,sort=True):
            prefix=dict(zip(REPORT_KEYS,key));calibration=cal_index[key];saved=quality_index.loc[key]
            threshold=report_cutoff(calibration.score_log,calibration.relErr)
            cutoff=-np.inf if saved.accept_none else float(saved.cutoff)
            if not (threshold==cutoff or np.isclose(threshold,cutoff,rtol=1e-10,atol=1e-12)):
                raise ValueError('Saved ML threshold does not reconstruct from calibration')
            gate=report_gate(part.relErr,part.score_log,cutoff)
            decisions=part.score_log.to_numpy()<=cutoff
            cal_decisions=calibration.score_log.to_numpy()<=cutoff
            if (not np.array_equal(decisions,part.accepted.to_numpy()) or gate['accepted_count']!=saved.accepted_count
                    or gate['violation_count']!=saved.violation_count or len(part)!=saved.test_count
                    or not np.array_equal(cal_decisions,calibration.accepted.to_numpy())
                    or int(cal_decisions.sum())!=saved.calibration_accepted_count
                    or int((cal_decisions&(calibration.relErr.to_numpy()>REPORT_EPSILON)).sum())!=saved.calibration_violation_count):
                raise ValueError('Saved ML decisions or counts do not match the original gate')
            with np.errstate(over='raise',invalid='raise'):
                prediction=np.exp(part.score_log.to_numpy())
                cal_prediction=np.exp(calibration.score_log.to_numpy())
            metric=report_metrics(part.relErr,part.score_log,prediction)
            constant=float(calibration.relErr.median())
            constant_mae=float(100*np.mean(np.abs(part.relErr.to_numpy()-constant)))
            metric.update(constant_cal_median_error=constant,constant_cal_median_test_mae_pp=constant_mae,
                          mae_gain_vs_constant_pp=constant_mae-metric['mae_pp'])
            metrics.append(dict(**prefix,score_space='predicted log-relative-error',numeric_error_comparable=True,**metric))
            gates.append(dict(**prefix,**gate,cutoff_source='original saved ML gate; calibration rechecked'))
            audits.append(dict(**prefix,calibration_reproduced=True,decisions_match=True,counts_match=True))
            curve=report_curve(part.relErr,part.score_log).assign(**prefix);curves.append(curve)
            for frame,score,pred,collection in ((part,part.score_log,prediction,test_parts),
                                                (calibration,calibration.score_log,cal_prediction,cal_parts)):
                output=frame[['qId','truth','estimate','relErr']].assign(**prefix,score=score.to_numpy(),
                    score_space='predicted log-relative-error',predicted_error=pred)
                output['accepted']=score.to_numpy()<=cutoff
                collection.append(output)
        missing=[]
        sa_test,sa_cal=report_sa_scores(source,cfg,test,cal,missing,state)
        if sa_test is not None:
            sa_cal_index={key:part for key,part in sa_cal.groupby(['seed','base_rate'],sort=True)}
            for (seed,rate),part in sa_test.groupby(['seed','base_rate'],sort=True):
                prefix=dict(seed=seed,base_rate=rate,variant='sa',validation_policy='not_applicable')
                calibration=sa_cal_index[(seed,rate)];cutoff=report_cutoff(calibration.score,calibration.relErr)
                metrics.append(dict(**prefix,score_space='raw SA heuristic',numeric_error_comparable=False,
                                    **report_metrics(part.relErr,part.score)))
                gates.append(dict(**prefix,**report_gate(part.relErr,part.score,cutoff),cutoff_source='matching outer calibration queries; same empirical rule'))
                curves.append(report_curve(part.relErr,part.score).assign(**prefix))
                test_parts.append(part.assign(accepted=part.score.to_numpy()<=cutoff))
                cal_parts.append(calibration.assign(accepted=calibration.score.to_numpy()<=cutoff))
        metrics=pd.DataFrame(metrics);gates=pd.DataFrame(gates);curve_table=pd.concat(curves,ignore_index=True)
        test_scores=pd.concat(test_parts,ignore_index=True);cal_scores=pd.concat(cal_parts,ignore_index=True)
        metrics.to_csv(destination/'all_query_metrics.csv',index=False);gates.to_csv(destination/'gate_results.csv',index=False)
        pd.DataFrame(audits).to_csv(destination/'gate_audit.csv',index=False)
        curve_table.to_csv(destination/'risk_coverage.csv',index=False)
        test_scores.to_csv(destination/'test_scores.csv',index=False);cal_scores.to_csv(destination/'calibration_scores.csv',index=False)
        pd.DataFrame(missing,columns=['input','path','reason']).to_csv(destination/'missing_inputs.csv',index=False)
        outer.to_csv(destination/'outer_splits.csv',index=False)
        summary=metrics.groupby(['validation_policy','variant','base_rate'],as_index=False).agg(
            settings=('seed','size'),test_case_rows=('test_count','sum'),mae_pp_mean=('mae_pp','mean'),
            p90_ae_pp_mean=('p90_ae_pp','mean'),bias_pp_mean=('bias_pp','mean'),log_mae_mean=('log_mae','mean'),
            constant_test_mae_pp_mean=('constant_cal_median_test_mae_pp','mean'),mae_gain_vs_constant_pp_mean=('mae_gain_vs_constant_pp','mean'),
            auc_bad_mean=('auc_bad','mean'),spearman_mean=('spearman_error_score','mean'))
        summary.to_csv(destination/'metrics_summary.csv',index=False)
        if (source/'models.csv').is_file():
            pd.read_csv(source/'models.csv').to_csv(destination/'models.csv',index=False)
            state['source_models_sha256']=report_sha(source/'models.csv')
        if sa_test is not None:
            sa_metrics=metrics[metrics.variant=='sa'][['seed','base_rate','auc_bad','spearman_error_score']]
            pair=metrics[metrics.variant!='sa'].merge(sa_metrics,on=['seed','base_rate'],suffixes=('_ml','_sa'),validate='many_to_one')
            pair['auc_delta']=pair.auc_bad_ml-pair.auc_bad_sa
            pair['spearman_delta']=pair.spearman_error_score_ml-pair.spearman_error_score_sa
            pair[['seed','validation_policy','variant','base_rate','auc_bad_ml','auc_bad_sa','auc_delta',
                  'spearman_error_score_ml','spearman_error_score_sa','spearman_delta']].to_csv(destination/'pairwise_vs_sa.csv',index=False)
        text=['# Full-query reliability evaluation','',
            'Primary numerical error metrics use EVERY test query, including rejected answers. MAE in percentage points '
            'is the average absolute difference between predicted and actual relative error; it is not histogram count accuracy.',
            'ML output is exp(score_log), without clipping to 100%. The training target was log(max(error,0.001)); '
            'log-space scores and linear-space MAE answer different questions. Underestimation and signed bias are reported.',
            'A simple numeric reference always predicts the matching calibration set median true error. '
            'constant_cal_median_test_mae_pp evaluates it on ALL test queries; positive mae_gain_vs_constant_pp means ML beats '
            'that constant reference. This does not turn the raw SA score into a numerical error estimate.',
            '','## All-query metrics','']
        for row in summary.itertuples():
            mae=f'{row.mae_pp_mean:.3f} percentage points' if math.isfinite(row.mae_pp_mean) else 'not defined for raw SA score'
            text.append(f'- {row.variant}, validation {row.validation_policy}, rate {row.base_rate:.8f}: '
                f'mean MAE {mae}; mean AUC {row.auc_bad_mean:.4f}; mean Spearman {row.spearman_mean:.4f}.')
        text+=['','## Fair comparisons and decision use','',
            'SA is a heuristic score, not a relative-error percentage prediction. Its numerical MAE is left blank. '
            'Compare SA and ML with full-test ranking metrics, complete risk–coverage curves and calibration-selected gate outcomes.',
            'The risk–coverage curve examines all score cutoffs on the test scores. Its thresholds are descriptive only. '
            'No test truth selects a new operational gate. Complete ties can produce jumps. At zero acceptance risk is undefined.',
            'gate_results.csv uses the ORIGINAL saved ML gates; SA gates use the same rule on matching calibration IDs. '
            'Read counts and acceptance together with violations. A few accepted answers with zero observed violations are not a guarantee.',
            '','## Scope and cost','',
            'The original 25% per-answer limit and 5% calibration target are experimental settings, not universal standards. '
            'The cohort excludes exact counts below 10, including zero. Source-cell splits are not strict geographic or cross-dataset transfer.',
            'All means across seeds are descriptive. Repeated query cases are not independent trials, and no significance claim is made. '
            'This test has already been inspected; final method claims require new held-out queries.',
            'Model fit time/tree count/file bytes are available in models.csv when supplied. Query-time SA/ML latency and E2E cost '
            'were not measured for these models. Notebook 12 timings must not be reused for notebook 13 models.',
            'The report reads saved predictions and verified logger scores only; no model is trained, no feature set is changed and no threshold is retuned.']
        if missing:text+=['','## Missing SA comparison','',
            'SA inputs are unavailable or incomplete. All ML metrics still use the complete original cohort. '
            'No subset or guessed SA scores were substituted. See missing_inputs.csv; provide the matching rows_file and rerun this report.']
        (destination/'analysis_en.md').write_text('\n'.join(text)+'\n')
        try:
            report_make_figures(destination,test_scores,curve_table,metrics,gates)
            state['figures']='complete'
        except ImportError as error:
            state['figures']='unavailable: '+str(error)
            print('Numerical report complete; plotting dependency unavailable:',error,flush=True)
        state.update(status='complete',analysis_status='complete' if sa_test is not None else 'partial_sa',
            ml_settings=int((metrics.variant!='sa').sum()),sa_settings=int((metrics.variant=='sa').sum()),
            ml_test_rows=len(test),reported_test_rows=len(test_scores),
            finished_utc=datetime.now(timezone.utc).isoformat())
        report_json(destination/'run.json',state);report_export(destination)
        print('FULL-QUERY REPORT COMPLETE:',destination,flush=True)
        print('SA comparison:',state['analysis_status'],flush=True)
        print('Upload:',destination/'full_query_report.zip',flush=True)
        return destination,metrics,gates
    except BaseException as error:
        state.update(status='failed',error=f'{type(error).__name__}: {error}')
        report_json(destination/'run.json',state)
        print('REPORT FAILED; diagnostics preserved:',destination,flush=True)
        raise


# RUN_FULL_REPORT
report_config=load_report_settings()
report_dir,all_query_metrics,gate_results=run_full_report(report_config)
display(all_query_metrics) if 'display' in globals() else print(all_query_metrics.to_string(index=False))